# Paper A: A Calibrated Placebo Test for Time-Varying Features in Credit Risk Models
**Muhammad Nawaz** · Replication notebook for the simulation part of the paper.

**How to use (plain language)**
1. Run the cells one by one from top to bottom (Shift+Enter).
2. By default the notebook runs a **quick test** (few repetitions per setting, a few minutes). The numbers will be noisy and will not match the paper exactly.
3. To reproduce the paper, set `REPS = 300` in the next cell. On our 2-core machine this took roughly 1.5 to 2 hours.
4. Needed packages: numpy, scipy, pandas, matplotlib, joblib (`pip install numpy scipy pandas matplotlib joblib`).

**What the notebook does:** it builds fake loan portfolios where we know the truth, adds a time series to the credit model, and checks how often three tests wrongly accept a series that is pure noise (and how often they find a real effect).

## 1. Settings

In [ ]:
REPS = 10        # repetitions per setting. Paper: 300 (roughly 1.5 to 2 hours on 2 cores). Quick test: 5 to 20 (a few minutes).
K = 99           # number of placebo series per test
B = 100          # bootstrap draws for the standard check
N_JOBS = 2       # CPU cores to use (set 1 if you have problems)
OUT_DIR = "."    # where results and figures are saved

## 2. Functions (the simulation and the three tests)

In [ ]:
import numpy as np, pandas as pd, time, os
from scipy.stats import rankdata
from joblib import Parallel, delayed

def auc(y, p):
    """Area under the ROC curve: how well scores p separate defaults (y=1) from non-defaults."""
    r = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def fit_logit(X, y, b0=None, it=8):
    """Logistic regression by Newton steps (a tiny ridge keeps it stable)."""
    b = np.zeros(X.shape[1]) if b0 is None else b0.copy()
    for _ in range(it):
        z = X @ b; p = 1 / (1 + np.exp(-z)); w = p * (1 - p) + 1e-9
        g = X.T @ (y - p) - 1e-4 * b
        H = (X * w[:, None]).T @ X + 1e-4 * np.eye(X.shape[1])
        step = np.linalg.solve(H, g); b += step
        if np.abs(step).max() < 1e-6: break
    return b

def ar1(T, phi, rng):
    """A standardised AR(1) series of length T; phi = how persistent (0 = no memory, 0.9 = slow)."""
    e = rng.standard_normal(T); z = np.zeros(T); z[0] = e[0]
    for t in range(1, T): z[t] = phi * z[t - 1] + e[t] * np.sqrt(1 - phi ** 2)
    return (z - z.mean()) / z.std()

def simulate(gamma, phi_s, rng, T=40, per_q=600, phi_c=0.9, sd_c=0.5):
    """40 quarters x 600 loans. Credit cycle c(t) is hidden. s(t) is the candidate series with true effect gamma."""
    n = T * per_q
    q = np.repeat(np.arange(T), per_q); yr = q // 4
    X = rng.standard_normal((n, 5)); beta = np.array([.8, -.6, .5, .4, -.3])
    c = sd_c * ar1(T, phi_c, rng)
    s = ar1(T, phi_s, rng)
    lin = -3.0 + X @ beta + c[q] + gamma * s[q]
    y = (rng.random(n) < 1 / (1 + np.exp(-lin))).astype(float)
    return X, y, q, yr, s

def one_rep(gamma, phi_s, seed, K=99, B=100, T=40, per_q=600, phi_c=0.9, sd_c=0.5, fe=True):
    """One simulated portfolio -> results of the standard check and the two placebo tests."""
    rng = np.random.default_rng(seed)
    X, y, q, yr, s = simulate(gamma, phi_s, rng, T, per_q, phi_c, sd_c)
    n = len(y)
    D = (yr[:, None] == np.arange(yr.max() + 1)[None, :]).astype(float)          # year dummies
    Xb = np.hstack([X, D]) if fe else np.hstack([X, np.ones((n, 1))])           # baseline model inputs
    idx = rng.permutation(n); ntr = int(.7 * n); tr, te = idx[:ntr], idx[ntr:]   # 70/30 random split
    b0 = fit_logit(Xb[tr], y[tr]); p0 = 1 / (1 + np.exp(-(Xb[te] @ b0))); yt = y[te]
    a0 = auc(yt, p0)
    def gain_of(series):
        v = series[q]
        if fe: v = v - np.array([v[yr == k].mean() for k in range(yr.max() + 1)])[yr]   # keep within-year part only
        Z = np.hstack([Xb, v[:, None]])
        b = fit_logit(Z[tr], y[tr], np.append(b0, 0.0))
        p = 1 / (1 + np.exp(-(Z[te] @ b)))
        return auc(yt, p) - a0, p
    g_real, p1 = gain_of(s)
    # (1) standard check: bootstrap the test loans only
    brng = np.random.default_rng(seed + 7); d = []
    for _ in range(B):
        i = brng.integers(0, len(te), len(te)); d.append(auc(yt[i], p1[i]) - auc(yt[i], p0[i]))
    std_pass = np.percentile(d, 2.5) > 0
    # (2) placebo test with shuffled series, (3) placebo test with persistence-matched series
    phi_hat = np.corrcoef(s[:-1], s[1:])[0, 1]
    g_iid = [gain_of(rng.permutation(s))[0] for _ in range(K)]
    g_ar = [gain_of(ar1(T, max(phi_hat, 0), rng))[0] for _ in range(K)]
    p_iid = (1 + sum(g >= g_real for g in g_iid)) / (K + 1)
    p_ar = (1 + sum(g >= g_real for g in g_ar)) / (K + 1)
    return dict(gamma=gamma, phi_s=phi_s, phi_c=phi_c, sd_c=sd_c, fe=int(fe), seed=seed, gain=g_real,
                std_pass=int(std_pass), p_iid=p_iid, p_ar=p_ar, phi_hat=phi_hat)

## 3. One test run (about 2 seconds)
A pure-noise, persistent series (true effect 0). `std_pass = 1` means the standard check wrongly accepted it. A placebo test accepts it only if its p-value is below 0.05.

In [ ]:
t0 = time.time()
print(one_rep(0.0, 0.9, 1, K=K, B=B)); print(f"{time.time()-t0:.1f} seconds")

## 4. Run all 14 settings of the paper
(gamma = true effect, phi_s = persistence of the series, phi_c / sd_c = credit cycle, fe = year effects in the model)

In [ ]:
# (gamma, phi_s, phi_c, sd_c, fe, first seed)
cells = [(0.0, 0.0, 0.9, 0.5, True, 10000), (0.0, 0.5, 0.9, 0.5, True, 10000), (0.0, 0.9, 0.9, 0.5, True, 10000),
         (0.1, 0.0, 0.9, 0.5, True, 10000), (0.1, 0.9, 0.9, 0.5, True, 10000), (0.2, 0.0, 0.9, 0.5, True, 10000),
         (0.2, 0.9, 0.9, 0.5, True, 10000), (0.3, 0.0, 0.9, 0.5, True, 10000), (0.3, 0.9, 0.9, 0.5, True, 10000),
         (0.0, 0.9, 0.9, 0.5, False, 20000), (0.0, 0.0, 0.9, 0.5, False, 20000),
         (0.0, 0.9, 0.99, 1.0, True, 20000), (0.0, 0.0, 0.99, 1.0, True, 20000), (0.0, 0.9, 0.99, 1.0, False, 20000)]
rows = []; t0 = time.time()
for g, ph, pc, sc, fe, s0 in cells:
    res = Parallel(n_jobs=N_JOBS)(delayed(one_rep)(g, ph, s0 + k, K, B, 40, 600, pc, sc, fe) for k in range(REPS))
    rows += res
    d = pd.DataFrame(res)
    print(f"effect={g} persistence={ph} cycle={pc} year_effects={fe}: standard {d.std_pass.mean():.2f} | shuffled {(d.p_iid<.05).mean():.2f} | persistence-matched {(d.p_ar<.05).mean():.2f}  ({time.time()-t0:.0f}s)", flush=True)
raw = pd.DataFrame(rows); raw.to_csv(os.path.join(OUT_DIR, "sim_raw_results.csv"), index=False)

## 5. Summarise: pass rates with 95% Wilson intervals

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h
out = []
for key, g in raw.groupby(["gamma", "phi_s", "phi_c", "sd_c", "fe"]):
    n = len(g); r = dict(zip(["gamma", "phi_s", "phi_c", "sd_c", "fe"], key)); r["n"] = n
    for name, k in [("std", g.std_pass.sum()), ("iid", (g.p_iid < 0.05).sum()), ("ar", (g.p_ar < 0.05).sum())]:
        lo, hi = wilson(k, n); r[name] = k / n; r[name + "_lo"] = lo; r[name + "_hi"] = hi
    out.append(r)
t = pd.DataFrame(out); t.to_csv(os.path.join(OUT_DIR, "sim_summary.csv"), index=False)
t.round(3)

## 6. Table 2 of the paper (persistent series 0.9, moderate cycle)

In [ ]:
sel = t[(t.phi_s == 0.9) & (t.phi_c == 0.9)].sort_values(["fe", "gamma"])
tab = sel[["fe", "gamma", "std", "iid", "ar"]].copy()
tab[["std", "iid", "ar"]] = (100 * tab[["std", "iid", "ar"]]).round(0)
tab.columns = ["year effects (1=yes)", "true effect", "standard check %", "shuffled placebo %", "persistence-matched %"]
tab

## 7. Figure 1: how often a useless series passes

In [ ]:
import matplotlib.pyplot as plt
BL, OR, GR = "#0072B2", "#D55E00", "#009E73"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
sc = [("Year effects,\nmoderate cycle,\niid series", (0.0, 0.0, 0.9, 1)), ("Year effects,\nmoderate cycle,\npersistent series", (0.0, 0.9, 0.9, 1)),
      ("Year effects,\ndrifting cycle,\npersistent series", (0.0, 0.9, 0.99, 1)), ("No period control,\nmoderate cycle,\npersistent series", (0.0, 0.9, 0.9, 0)),
      ("No period control,\ndrifting cycle,\npersistent series", (0.0, 0.9, 0.99, 0))]
fig, ax = plt.subplots(figsize=(7.2, 3.5)); w = 0.26
for j, (col, lab, c) in enumerate([("std", "Standard check (bootstrap)", OR), ("iid", "Placebo test, shuffled series", GR), ("ar", "Placebo test, persistence-matched", BL)]):
    vals, lo, hi = [], [], []
    for _, key in sc:
        r = t[(t.gamma == key[0]) & (t.phi_s == key[1]) & (t.phi_c == key[2]) & (t.fe == key[3])].iloc[0]
        vals.append(100 * r[col]); lo.append(100 * (r[col] - r[col + "_lo"])); hi.append(100 * (r[col + "_hi"] - r[col]))
    ax.bar([i + (j - 1) * w for i in range(len(sc))], vals, w, color=c, label=lab, yerr=[lo, hi], error_kw=dict(lw=0.8, capsize=2, ecolor="#444"))
ax.axhline(5, color="#555", lw=0.8, ls="--"); ax.text(2.5, 6.5, "intended 5%", ha="center", fontsize=8, color="#555")
ax.set_xticks(range(len(sc))); ax.set_xticklabels([s[0] for s in sc], fontsize=7.5)
ax.set_ylabel("Share of useless series that pass (%)"); ax.legend(frameon=False, fontsize=8, loc="upper left"); ax.set_ylim(0, 80)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig1.png"), dpi=200); plt.show()

## 8. Figure 2: how often a real effect is found (power)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.1), sharey=True)
for ax, ps, tt in zip(axs, [0.0, 0.9], ["Non-persistent series", "Persistent series (phi = 0.9)"]):
    for col, lab, c in [("std", "Standard check", OR), ("iid", "Placebo, shuffled", GR), ("ar", "Placebo, persistence-matched", BL)]:
        d = t[(t.phi_s == ps) & (t.phi_c == 0.9) & (t.fe == 1)].sort_values("gamma")
        ax.plot(d.gamma, 100 * d[col], marker="o", color=c, lw=1.6, label=lab)
    ax.set_title(tt, fontsize=9); ax.set_xlabel("Size of the true effect"); ax.set_xticks([0, .1, .2, .3])
axs[0].set_ylabel("Share of real effects detected (%)"); axs[0].legend(frameon=False, fontsize=7.5, loc="upper left")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig2.png"), dpi=200); plt.show()

## 9. Real-data part (not in this notebook)
The Lending Club and Bondora runs need the data and tone files of the companion paper (SSRN 7622180), which are not redistributed here. Their results are reported in Table 3 of the paper; the script `real.py` in the replication package reproduces them when the data are available.

## 10. Disclosure of AI use
Claude (Anthropic), a generative AI assistant, was used for language correction, for searching relevant literature, and for drafting and correcting the Python code. The author ran and checked the code and the results, and takes full responsibility for the content.